In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

df = pd.DataFrame(dataset["train"])

df.head()

In [ ]:
print("Total de reseñas:", len(df))

print("\nDistribución de sentimientos:")
print(df["label"].value_counts())
print("\n0 = negativa, 1 = positiva")

In [ ]:
X = df["text"]    # las reseñas
y = df["label"]   # 0 = negativa, 1 = positiva

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Reseñas para entrenar:", len(X_train))
print("Reseñas para evaluar:", len(X_test))

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

print("Forma de los datos de entrenamiento:", X_train_vec.shape)
print("Cada reseña ahora es un vector de", X_train_vec.shape[1], "números")

In [ ]:
modelo = LogisticRegression(max_iter=1000)

modelo.fit(X_train_vec, y_train)

print("Modelo entrenado exitosamente")

In [ ]:
# El modelo predice el sentimiento de las reseñas que nunca vio
y_pred = modelo.predict(X_test_vec)

# Accuracy: porcentaje de reseñas clasificadas correctamente
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}")

# Matriz de confusión: muestra dónde acierta y dónde se equivoca
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negativa", "Positiva"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

In [ ]:
# Tomamos ejemplos reales del conjunto de prueba
ejemplos = X_test.reset_index(drop=True)
etiquetas_reales = y_test.reset_index(drop=True)
predicciones = modelo.predict(X_test_vec)

def mostrar_ejemplo(indice, tipo):
    texto = ejemplos[indice]
    real = "Positiva" if etiquetas_reales[indice] == 1 else "Negativa"
    pred = "Positiva" if predicciones[indice] == 1 else "Negativa"
    resultado = "CORRECTO" if real == pred else "INCORRECTO"
    print(f"--- Ejemplo {tipo} ---")
    print(f"Reseña: {texto}")
    print(f"Etiqueta real: {real}")
    print(f"Predicción:    {pred}")
    print(f"Resultado:     {resultado}")
    print()

# 1. Bien clasificado — busca el primero donde acertó
for i in range(len(predicciones)):
    if predicciones[i] == etiquetas_reales[i]:
        mostrar_ejemplo(i, "bien clasificado")
        break

# 2. Mal clasificado — busca el primero donde se equivocó
for i in range(len(predicciones)):
    if predicciones[i] != etiquetas_reales[i]:
        mostrar_ejemplo(i, "mal clasificado")
        break

# 3. Dudoso — probabilidad más cercana a 0.5
probabilidades = modelo.predict_proba(X_test_vec)
idx_dudoso = abs(probabilidades[:, 1] - 0.5).argmin()
mostrar_ejemplo(idx_dudoso, "dudoso (ambiguo)")